# Exploring some basics from the MCS Zooms
## Written by Eric Rohr

In [1]:
### import modules
import illustris_python as il # type: ignore
import matplotlib.pyplot as plt 
import numpy as np 
import matplotlib as mpl 
import matplotlib.cm as cm 
import matplotlib.patheffects as pe 
import matplotlib.transforms as transforms  
from matplotlib.gridspec import GridSpec  
import matplotlib.gridspec as gridspec  
from matplotlib.patches import Patch  
import matplotlib.patches as patches  
from mpl_toolkits.axes_grid1.inset_locator import inset_axes  
from mpl_toolkits.axes_grid1 import make_axes_locatable  
from scipy.ndimage import gaussian_filter  
from scipy import ndimage  
from scipy.interpolate import interp1d  
from scipy import interpolate  
from temet.util.sphMap import sphMap  #type: ignore
import scipy.stats  
from scipy.stats import norm  
from sklearn.neighbors import KernelDensity  
from scipy.stats import ks_2samp, anderson_ksamp  
from scipy.optimize import curve_fit  
from astropy.cosmology import Cosmology, FlatLambdaCDM, z_at_value
from astropy import units as u 
from astropy import constants as const 
import os
import glob
import csv
from pathlib import Path
import time
import h5py  
import rohr_utils as ru 
import utils.io as io
from utils.units import *
import random
import six  
import scida
from scida import load
import pint 
from createMCSTFiles import createMCSTFiles
import createOffsets
from stellar_array_helpers import expand_all_arrays

%matplotlib inline

plt.style.use('fullpage.mplstyle')

os.chdir('/u/reric/Scripts/')
! pwd



Warning! Using default configuration. Please adjust/replace in '/u/reric/.config/scida/config.yaml'.
/vera/u/reric/Scripts


In [5]:
def prepareSim(simFamily, simName, globalStartPath='/virgotng/universe', localSimFamilyPath='../'):
    """
    Prepare the given simulation for analysis. Returns the basePath.
    """
    basePath = createSimDirecStruct(simFamily, simName, globalStartPath=globalStartPath, localSimFamilyPath=localSimFamilyPath)
    io.createSnapTimes(basePath)
    snapTimes = io.loadSnapTimes(basePath)
    for snapNum in snapTimes['SnapNum']:
        createOffsets.createOffsets(basePath, snapNum)

    return basePath


def createSimDirecStruct(simFamily, simName, globalStartPath='/virgotng/universe', localSimFamilyPath='../'):
    """
    create a local copy using symbolic links to a given simulation.
    the default global path is /virgotng/universe, which then gets
    combined with simFamily (and output) to become the global basePath.
    localSimFamilyPath, which defaults to the parent directory, will 
    then hold the simFamily directory, which holds simName. Lastly, 
    local basePath = localSimFamilyPath + simFamily + simName + output.
    if local simFamily directory already exists, then nothing is done.
    Returns the local basePath.
    """

    localSimPath = os.path.join(localSimFamilyPath, 'sims.' + simFamily, simName)
    localbasePath = os.path.join(localSimPath, 'output')
    if os.path.isdir(localSimPath):
        return localbasePath
    else:
        os.makedirs(localSimPath)

    # output directory should be a sym link to the global direc
    globalBasePath = os.path.join(globalStartPath, simFamily, simName, 'output')
    os.symlink(globalBasePath, localbasePath, target_is_directory=True)

    # postprocesing directory should be local, but existing catalogs should be linked
    localPostprocessingPath = os.path.join(localSimPath, 'postprocessing')
    os.makedirs(localPostprocessingPath)

    globalPostprocessingPath = os.path.join(Path(globalBasePath).parent, 'postprocessing')
    os.system('ln -s %s %s'%(os.path.join(globalPostprocessingPath, '*'), os.path.join(localPostprocessingPath, '.')))

    return localbasePath

    

In [6]:
inputs = dict(IllustrisTNG=['TNG100-1', 'TNG100-3'],
              Eagle=['Eagle100-1'],
              Illustris=['Illustris-1', 'Illustris-3'],
              Simba=['Simba100-1'])

Sims = {}

for simFamily in inputs:
    simNames = inputs[simFamily]
    for simName in simNames:
        print(simFamily, simName)
        Sims[simName] = dict(basePath=prepareSim(simFamily, simName))


IllustrisTNG TNG100-1
IllustrisTNG TNG100-3
Eagle Eagle100-1
Illustris Illustris-1
Illustris Illustris-3
Simba Simba100-1


In [ ]:
basePath = '../sims.Illustris/Illustris-1/output'
snapNum = 99
Header = io.loadHeader(basePath, snapNum)
Header

In [ ]:
keys = ['HubbleParam', 'Omega0', 'OmegaBaryon']
if all([key in Header for key in keys]):
    print('True')
else:
    print('False')

In [ ]:
[key in Header for key in keys]

In [ ]:
def createSimName(baseSim='L35n2160TNG', haloID=31619, targetRedshift=3, level=15, model='ST8', TNG50=False):
    """Create the simulation name from given parameters"""
    if TNG50:
        return 'TNG50-1'
    return '%s_h%d_z%d_L%d_%s'%(baseSim, haloID, targetRedshift, level, model)


def return_basePath(sim):
    """ return basePath for given simulation """
    valid_sims = ['L35n2160TNG_h31619_z3_L13_ST8',
                  'L35n2160TNG_h31619_z3_L14_ST8',
                  'L35n2160TNG_h31619_z3_L15_ST8',
                  'L35n2160TNG_h31619_z3_L13_ST8m',
                  'L35n2160TNG_h31619_z3_L14_ST8m',
                  'L35n2160TNG_h31619_z3_L15_ST8m',
                  'L35n2160TNG_h31619_z3_L13_ST8s',
                  'L35n2160TNG_h31619_z3_L14_ST8s',
                  'L35n2160TNG_h31619_z3_L15_ST8s',
                  'L35n2160TNG_h31619_z3_L15_TNG',
                  'L35n2160TNG_h31619_z3_L11_TNG',
                  'L35n2160TNG_h4182_z3_L14_ST8',
                  'TNG50-1']
    if sim in valid_sims:
        if sim == 'TNG50-1':
            return './../sims.IllustrisTNG/%s/output'%sim
        return os.path.join('./../sims.MCST', sim, 'output')
    else:
        raise ValueError('sim %s not found'%sim)


def findSnapNum(basePath, key='Redshift', val=5.0):
    """Find the snapshot number closest to the given key, value pair"""

    snapTimes = io.loadSnapTimes(basePath)
    index = np.argmin(np.abs(snapTimes[key] - val))
    closest_val = snapTimes[key][index]
    if np.abs(closest_val - val) / val > 0.01:
        raise RuntimeWarning('No snapshot found with %s = %.3f. Closest value is %.3f.'%(key, val, closest_val))
    return snapTimes['SnapNum'][index]




In [ ]:
class Sim: 
    """Create a class for the given simulation"""

    def __init__(self, baseSim='L35n2160TNG', targetHaloID=31619, targetRedshift=3, level=15, model='ST8', key='Redshift', val=5.0, subfindID=0, TNG50=False,
                 addMCSTFiles=True, addSubhaloGas=True, addHalo=True, addSubhalo=True):
        """ initialize the class. Note that the TNG50 flag is only to be used for the original TNG50 simulation"""

        kwargs = locals().copy()
        for _key in kwargs:
            if _key != 'self':
                setattr(self, _key, kwargs[_key])

        self.sim = createSimName(self.baseSim, self.targetHaloID, self.targetRedshift, self.level, self.model, self.TNG50)
        self.basePath = return_basePath(self.sim)
        self.snapNum = findSnapNum(self.basePath, self.key, self.val)
        self.Header = io.loadHeader(self.basePath, self.snapNum)
        self.Parameters = io.loadParameters(self.basePath, self.snapNum)
        self.Config = io.loadConfig(self.basePath, self.snapNum)
        self.snapTimes = io.loadSnapTimes(self.basePath)
        index = self.snapTimes['SnapNum'] == self.snapNum
        self.Redshift = self.snapTimes['Redshift'][index][0]
        self.Time = self.snapTimes['Time'][index][0]
        self.CosmicTime = self.snapTimes['CosmicTime'][index][0]

        # if original TNG50 sim, use merger trees to find the subhaloID and FoF host  
        if TNG50:
            self.model = 'TNG'
            snapNum_target = findSnapNum(self.basePath, val=targetRedshift)
            Halo_targetRedshift = il.groupcat.loadSingle(self.basePath, snapNum_target, haloID=self.targetHaloID)
            tree = io.loadMainTreeBranch(self.basePath, snapNum_target, subfindID=Halo_targetRedshift['GroupFirstSub'], fields=['SnapNum', 'SubfindID'])
            snapNum_interest = findSnapNum(self.basePath, val=val)
            subfindID_interest = tree['SubfindID'][tree['SnapNum'] == snapNum_interest][0]
            self.snapNum = snapNum_interest
            self.subfindID = subfindID_interest

        # check if offsets already exists, and if not, then create 
        createOffsets.createOffsets(self.basePath, self.snapNum)

        if addSubhalo:
            self.Subhalo = il.groupcat.loadSingle(self.basePath, self.snapNum, subhaloID=self.subfindID)
            io.convertGroupUnits(self.basePath, self.snapNum, self.Subhalo)
        if addHalo:
            self.Halo = il.groupcat.loadSingle(self.basePath, self.snapNum, haloID=int(self.Subhalo['SubhaloGrNr']))
            io.convertGroupUnits(self.basePath, self.snapNum, self.Halo)
        if addSubhaloGas:
            self.SubhaloGas = il.snapshot.loadSubhalo(self.basePath, self.snapNum, self.subfindID, partType=0)
            io.convertSnapshotUnits(self.basePath, self.snapNum, self.SubhaloGas)        
            # postprocess extra datasets
            io.computeCellSizes(self.SubhaloGas, self.basePath, self.snapNum)
            io.computeGasPressure(self.SubhaloGas, self.basePath, self.snapNum)
            io.computeJeansNumber(self.SubhaloGas, self.basePath, self.snapNum)      
            # add postprocessed datasets for TNG sims
            if self.model == 'TNG':
                if self.SubhaloGas['count'] > 0:
                    io.computeTemperature(self.SubhaloGas)
                    io.computeCoolingTime(self.SubhaloGas, self.basePath, self.snapNum)  

        # add MCST sf_ and sn_details files
        if (addMCSTFiles) and (self.model != 'TNG'):
            self.MCSTFiles = io.loadMCSTFiles(self.basePath, self.snapNum)

        # define short titles for plotting
        self.plotTitle = r'%s L%d'%(self.model, self.level)
        if TNG50:
            self.plotTitle = r'TNG50 (L11)'
        add_kwargs(self)


def add_kwargs(Sim, **kwargs):
    """ add kwargs to class"""
    max_level = 16
    min_level = 11
    if Sim.TNG50:
        kwargs['cmap'] = mpl.colormaps['Greys'].copy()
    elif Sim.model == 'TNG':
        kwargs['cmap'] = mpl.colormaps['YlGn'].copy()
    elif Sim.model == 'ST8':
        kwargs['cmap'] = mpl.colormaps['RdPu'].copy()
    elif Sim.model == 'ST8m':
        kwargs['cmap'] = mpl.colormaps['GnBu'].copy()
    elif Sim.model == 'ST8s':
        kwargs['cmap'] = mpl.colormaps['YlGn'].copy()

    if Sim.level == 15:
        kwargs['ls'] = kwargs['linestyle'] = '-'
        kwargs['lw'] = kwargs['linewidth'] = 2.5
        kwargs['hatch'] = '/'
    elif Sim.level == 14:
        kwargs['ls'] = kwargs['linestyle'] = '--'
        kwargs['lw'] = kwargs['linewidth'] = 2.0
        kwargs['hatch'] = '\\'
    elif Sim.level <= 13:
        kwargs['ls'] = kwargs['linestyle'] = ':'
        kwargs['lw'] = kwargs['linewidth'] = 1.5
        kwargs['hatch'] = 'o'

    level = Sim.level
    if level <= 13:
        level = 13
    
    kwargs['c'] = kwargs['color'] = kwargs['cmap']((level - min_level) / (max_level - min_level))

    Sim.kwargs = kwargs
    return



In [ ]:
L35n2160TNG_h31619_z3_L15_ST8 = Sim(model='ST8')
L35n2160TNG_h31619_z3_L14_ST8 = Sim(model='ST8', level=14)
L35n2160TNG_h31619_z3_L13_ST8 = Sim(model='ST8', level=13)
L35n2160TNG_h31619_z3_L15_TNG = Sim(model='TNG')
L35n2160TNG_h31619_z3_L11_TNG = Sim(model='TNG', level=11)
L35n2160TNG_h31619_z3_L15_ST8m = Sim(model='ST8m', val=8.0)
L35n2160TNG_h31619_z3_L14_ST8m = Sim(model='ST8m', level=14)
L35n2160TNG_h31619_z3_L13_ST8m = Sim(model='ST8m', level=13)
L35n2160TNG_h31619_z3_L15_ST8s = Sim(model='ST8s', val=5.5)
L35n2160TNG_h31619_z3_L14_ST8s = Sim(model='ST8s', level=14)
L35n2160TNG_h31619_z3_L13_ST8s = Sim(model='ST8s', level=13)
TNG50_h31619_z3 = Sim(TNG50=True)

Sims = [L35n2160TNG_h31619_z3_L15_ST8,
        L35n2160TNG_h31619_z3_L14_ST8,
        L35n2160TNG_h31619_z3_L13_ST8,
        L35n2160TNG_h31619_z3_L15_TNG,
        L35n2160TNG_h31619_z3_L11_TNG,
        TNG50_h31619_z3]

In [ ]:
out_direc = '../Figures/MCST/'
savefig = False

In [ ]:
# Metallicity, GrackleCoolTime, and GrackleTemperature have different keys for TNG sims
dset_swap = dict(Metallicity='GFM_Metallicity', GrackleCoolTime='CoolingTime', GrackleTemperature='Temperature',
                 GFM_Metallicity='Metallicity', CoolingTime='GrackleCoolTime', Temperature='GrackleTemperature')
mask_keys = ['HeatingUp', 'CoolingDown']

def plot_nTphasespace(Sim, weights='Masses', x_key='Density', y_key='GrackleTemperature', savefig=False, mask=None, norm=True,
                      add_Nj=True, Nj=[0.5, 1.0, 2.0, 4.0, 8.0]):
    """
    Plot number density vs temperature phase space, with optional weights.
    Optionally can only consider certain gas cells based on mask, where 
    currntly accepted values are 'HeatingUp' and 'CoolingDown'.
    Defaults to include lines of constant Jeans number.
    """

    if mask and mask not in mask_keys:
        raise ValueError('mask %s is currently not supported'%mask)

    fig, ax = plt.subplots(figsize=(7,5))

    SubhaloGas = Sim.SubhaloGas

    x_min = -4.0
    x_max = 7.0
    x_max = 10.0
    y_min = 1.0
    y_max = 6.5
    nbins = 200
    x_bins = np.linspace(x_min, x_max, nbins)
    y_bins = np.linspace(y_min, y_max, nbins)
    bins = [y_bins, x_bins]

    _Density = SubhaloGas['Density'].to(const.m_p/u.cm**3)

    text = 'Subhalo Gas'
    if not mask:
        _mask = np.ones_like(_Density.value, dtype=bool)
    else:
        if mask == mask_keys[0] or mask == mask_keys[1]:
            mask_key = 'GrackleCoolTime'
            if mask_key not in SubhaloGas:
                if mask_key in dset_swap:
                    mask_key = dset_swap[mask_key]
            _dset = SubhaloGas[mask_key]
            if mask == mask_keys[0]:
                _mask = _dset.value >= 0
                text += '\nHeating Up'
            else:
                _mask = _dset.value < 0
                text += '\nCooling Down'
    ax.text(0.05, 0.95, text, ha='left', va='top', transform=ax.transAxes)

    Density = _Density[_mask].value
    y = SubhaloGas[y_key][_mask].value
    Masses = SubhaloGas['Masses'][_mask].value

    stat_mass = scipy.stats.binned_statistic_2d(np.log10(y), np.log10(Density), Masses, statistic='sum', bins=bins)[0]

    if weights == 'Masses':
        if norm:
            stat = stat_mass / Masses.sum()
            cbar_label = r'Gas Mass Fraction'
            cbar_norm = mpl.colors.LogNorm(vmin=1.1e-7, vmax=9.9e-3)
        else:
            stat = stat_mass
            cbar_label = r'Gas Mass [%s]'%SubhaloGas['Masses'].unit.to_string('latex_inline')
            cbar_norm = mpl.colors.LogNorm(vmin=2e1, vmax=3.0e6)
        cmap = cm.viridis.copy()
        stat[(stat > 0) & (stat < cbar_norm.vmin)] = cbar_norm.vmin

    else:
        if weights not in SubhaloGas:
            if weights in dset_swap:
                weights = dset_swap[mask_key]
            else:
                raise ValueError('Error: %s not in SubhaloGas'%weights)
        _weights = SubhaloGas[weights][_mask]
        if mask == 'CoolingDown':
            _weights *= -1

        stat_weight = scipy.stats.binned_statistic_2d(np.log10(y), np.log10(Density), _weights.value * Masses, statistic='sum', bins=bins)[0]
        stat_mask = (stat_mass > 0) & (stat_weight > 0)
        stat = np.zeros_like(stat_mass)
        stat[stat_mask] = stat_weight[stat_mask] / stat_mass[stat_mask]
        cbar_label = r'%s [%s]'%(weights.replace('_', '\_'), _weights.unit.to_string('latex_inline'))
        cmap = cm.magma.copy()    
        if weights in ['Metallicity', 'GFM_Metallicity']:
            cbar_norm = mpl.colors.LogNorm(vmin=3.0e-5, vmax=3.0)
            cbar_norm = mpl.colors.LogNorm(vmin=2.0e-3, vmax=3.0)
            stat[stat_mask & (stat < cbar_norm.vmin)] = cbar_norm.vmin
        elif weights in ['CoolingTime', 'GrackleCoolTime']:
            if not mask:
                raise ValueError('Key %s requires a mask'%weights)
            cbar_norm = mpl.colors.LogNorm(3.0e1, 3.0e8)
        else:
            cbar_norm = mpl.colors.LogNorm()

    if np.any(np.isnan(stat)):
        raise ValueError('Error: some values are NaN')

    cmap.set_under('white', 1.)
    cmap.set_bad('white', 1.)

    extent = [x_min,x_max, y_min, y_max]
    origin = 'lower'
    kwargs = dict(cmap=cmap, extent=extent, origin=origin, norm=cbar_norm, rasterized=False)

    h = ax.imshow(stat, **kwargs)

    ax.set_ylabel(r'Gas Temperature $[\log_{10} {\rm K}]$')
    ax.set_xlabel(r'Gas Density $[\log_{10} (n = \rho / {\rm m_H}) \, {\rm cm^{-3}}]$')

    cax = ax.inset_axes(bounds=[0.55, 0.9, 0.4, 0.05])
    #cax = inset_axes(ax, width="40%", height="5%", loc='upper right', bbox_to_anchor=(0.05, 0.05, 0.95, 0.95), bbox_transform=ax.transAxes)
    cbar = fig.colorbar(h, cax=cax, orientation='horizontal')
    cbar.set_label(cbar_label)

    text = r'h%d $z=%.1f$'%(Sim.targetHaloID, Sim.Redshift) + '\n%s'%Sim.plotTitle
    ax.text(0.05, 0.05, text, transform=ax.transAxes, ha='left', va='bottom', ma='left')

    # add lines of constant Jeans number
    if add_Nj:
        TracerMass = (Sim.Header['MassTable'][3] / Sim.Header['HubbleParam'] * code_mass).to(standard_mass)
        TracerMCPerCell = Sim.Parameters['TracerMCPerCell']
        TargetGasMass = TracerMass * TracerMCPerCell

        ax.set_xlim(ax.get_xlim())
        ax.set_ylim(ax.get_ylim())

        rho = (10.**(np.array(ax.get_xlim())) * (const.m_p / u.cm**3)).to(const.m_p / u.cm**3)

        Nj_kwargs = dict(color='k', ls='-', lw=1, marker='None')
        for Nj_i, _Nj in enumerate(Nj):
            Tj = compute_JeansTemperature(rho, TargetGasMass, _Nj)
            ax.plot(np.log10(rho.value), np.log10(Tj.value), **Nj_kwargs)
            # label the first and last lines
            if Nj_i == 0 or Nj_i == (len(Nj) - 1):
                x_text = (ax.get_xlim()[1]) * 0.9
                y_text = np.log10(compute_JeansTemperature(10.**(x_text) * (const.m_p / u.cm**3).to(const.m_p / u.cm**3), TargetGasMass, _Nj).value)
                angle = np.degrees(np.arctan2(y_text - np.log10(Tj.value[0]), x_text - np.log10(rho.value[0])))
                va = 'bottom'
                if Nj_i == 0:
                    va = 'top'
                ax.text(x_text, y_text, r'$N_j = %.1f$'%_Nj, ha='center', va=va, rotation=angle)

    if savefig:
        out_fname = 'nTphasespace_%s_z%.1f_%s.pdf'%(Sim.sim, Sim.Redshift, weights)
        if norm and weights == 'Masses':
            out_fname = 'nTphasespace_%s_z%.1f_%s_norm.pdf'%(Sim.sim, Sim.Redshift, weights)
        if mask:
            out_fname = 'nTphasespace_%s_z%.1f_%s_%s.pdf'%(Sim.sim, Sim.Redshift, weights, mask)

        fig.savefig(os.path.join(out_direc, out_fname), bbox_inches='tight')

    return fig, ax


def compute_JeansTemperature(rho, mgas, Nj):
    """ Return the temperature [K] of a gas cell with a given Jeans Number Nj, density rho, and gas mass mgas """
    mu = 50. / 41. # neutral, metal-free gas
    gamma = 5. / 3. # adiabatic index
    prefactor = (mu * const.m_p * const.G) / ((gamma - 1) * const.k_B)
    return (prefactor * (6. * np.pi**(-5./2.) * rho**(1./2.) * Nj * mgas)**(2./3.)).to('K')

    

    

In [ ]:
baseSim = Sims[0]
weights = 'Metallicity'
_outdirec = os.path.join(out_direc, 'PhaseSpaceDiagrams', baseSim.sim, weights)

if savefig:

    if not os.path.exists(_outdirec):
        os.makedirs(_outdirec)

    files = glob.glob(os.path.join(baseSim.basePath, 'snapdir*'))
    files.sort()
    for file_i, file in enumerate(files):
        snapNum = int(file[-3:])
        _Sim = Sim(key='SnapNum', val=snapNum)
        fig, ax = plot_nTphasespace(_Sim, weights=weights)
        fname = 'PhaseSpaceDiagram_%s_%s_snapNum%03d.png'%(baseSim.sim, weights, _Sim.snapNum)
        fig.savefig(os.path.join(_outdirec, fname), bbox_inches='tight', dpi=300)
        plt.close(fig)


In [ ]:
for weights in ['Masses']:
    for _Sim in Sims:
        y_key='GrackleTemperature'
        if _Sim.model == 'TNG':
            y_key='Temperature'
        _, _ = plot_nTphasespace(_Sim, y_key=y_key, savefig=savefig, weights=weights)

In [ ]:
for _Sim in Sims:
    y_key='GrackleTemperature'
    weights='Metallicity'
    if _Sim.model == 'TNG':
        y_key='Temperature'
        weights='GFM_Metallicity'
    _, _ = plot_nTphasespace(_Sim, y_key=y_key, weights=weights, savefig=savefig)

In [ ]:
_, _ = plot_nTphasespace(Sims[0], weights='Masses', mask=None, norm=False, savefig=savefig)
_, _ = plot_nTphasespace(Sims[0], weights='Masses', mask='HeatingUp', norm=False, savefig=savefig)
_, _ = plot_nTphasespace(Sims[0], weights='Masses', mask='CoolingDown', norm=False, savefig=savefig)

fig, ax = plot_nTphasespace(Sims[0], weights='GrackleCoolTime', mask='HeatingUp', savefig=savefig)
fig, ax = plot_nTphasespace(Sims[0], weights='GrackleCoolTime', mask='CoolingDown', savefig=savefig)

In [ ]:


def plot_HistogramComparison(Sims, dset_key='Masses', savefig=False, dset_log=True, nbins=50, xlabel=None, mass_weighted=True, addSFingGas=False):
    """Plot the 1D distribution of a given dataset across Sims"""

    # check if Sims is a singular instance of the class Sim
    if isinstance(Sims, Sim):
        Sims = [Sims]

    # start figure
    fig, ax = plt.subplots()
    ax.set_ylim(5.0e1, 2.0e8)
    ax.set_yscale('log')
    if mass_weighted:
        ylabel = r'Gas Mass per %s bin [%s]'%(dset_key.replace('_', '\_'), Sims[0].SubhaloGas['Masses'].unit.to_string('latex_inline'))
    else:
        ylabel = r'Number of Gas Cells per %s bin'%(dset_key)
    ax.set_ylabel(ylabel)
    if xlabel:
        ax.set_xlabel(xlabel)
    else:
        if dset_key == 'Density':
            ax.set_xlabel(r'Gas Density $[\log_{10} (n = \rho / {\rm m_H}) \, {\rm cm^{-3}}]$')        
        else:
            ax.set_xlabel(r'%s [$\log_{10}$ %s]'%(dset_key.replace('_', '\_'), Sims[0].SubhaloGas[dset_key].unit.to_string('latex_inline')))

    for _Sim_i, _Sim in enumerate(Sims):
        # currently only works for zoom simulations
        #if _Sim.TNG50:
        #    raise ValueError('Currently not supported for TNG50')
    
        if dset_key not in _Sim.SubhaloGas:
            if dset_key in dset_swap:
                dset_key = dset_swap[dset_key]
            else:
                raise KeyError('dset_key %s not in _Sim.SubhaloGas or dset_swap'%dset_key)

        # find the min and max values across all sims to set the bins
        _dset = _Sim.SubhaloGas[dset_key].copy()
        if dset_key == 'Density':
            _dset = _dset.to(const.m_p / u.cm**3)

        if dset_log: 
            dset = np.log10(_dset.value)
        else:
            dset = _dset.value

        if _Sim_i == 0:
            bin_min = dset.min()
            bin_max = dset.max()
        else:
            if dset.min() < bin_min:
                bin_min = dset.min()
            if dset.max() > bin_max:
                bin_max = dset.max()
        
    # add a small buffer on either side, which should work well if dset_log
    bin_min -= 0.5 
    bin_max += 0.5

    bins = np.linspace(bin_min, bin_max, nbins)
    bin_width = bins[1] - bins[0]

    for _Sim_i, _Sim in enumerate(Sims):
        _SubhaloGas = _Sim.SubhaloGas   
        if dset_key not in _Sim.SubhaloGas:
            if dset_key in dset_swap:
                dset_key = dset_swap[dset_key]
        _dset = _SubhaloGas[dset_key]

        if dset_key == 'Density':
            _dset = _dset.to(const.m_p / u.cm**3)

        if dset_log: 
            dset = np.log10(_dset.value)
        else:
            dset = _dset.value

        # make extra histograms for only SFing Gas?
        masks = [np.ones_like(dset, dtype=bool)]
        if addSFingGas:
            masks.append(_SubhaloGas['StarFormationRate'].value > 0)
            if _Sim_i == 0:
                ax.hist(np.zeros(0), bins=bins, histtype='stepfilled', color='tab:gray', label='SFing Gas', edgecolor='k')

        if mass_weighted:
            weights = _SubhaloGas['Masses'].value
        else:
            weights = np.ones_like(dset)
        
        for mask_i, mask in enumerate(masks):
            _kwargs = _Sim.kwargs.copy()
            # offset bins slightly for visual clarity
            _bins = bins + bin_width * 0.1 * _Sim_i
            kwargs = dict(weights=weights[mask], bins=_bins, histtype='step', label=_Sim.plotTitle,
                        color=_kwargs['c'], linewidth=_kwargs['lw'], linestyle=_kwargs['ls'])
            if mask_i == 1:
                kwargs.update(dict(alpha=0.25, histtype='stepfilled', hatch=_kwargs['hatch'], label=None, edgecolor='k'))
            ax.hist(dset[mask], **kwargs)
        # add vertical line at minimum hydro softening
        if dset_key == 'CellSizes':
            min_softening = np.log10(_Sim.Parameters['MinimumComovingHydroSoftening'] * _Sim.Header['Time'] / _Sim.Header['HubbleParam'])
            if _Sim.model == 'TNG' and _Sim.level == 15:
                min_softening += 0.05
            ax.vlines(min_softening, ymin=ax.get_ylim()[0], ymax=ax.get_ylim()[1], colors=_kwargs['c'], lw=1.5, zorder=1, alpha=1.0)

    ax.legend(title='h%s $z=%.1f$'%(_Sim.targetHaloID, _Sim.Redshift), loc='upper left')

    text = 'All Subhalo Gas'
    ax.text(0.975, 0.975, text, transform=ax.transAxes, ha='right', va='top', ma='right')

    if savefig:
        out_fname = 'HistogramComparison_%s_h%d_z%d_z%.1f_%s.pdf'%(_Sim.baseSim, _Sim.targetHaloID, _Sim.targetRedshift, _Sim.Redshift, dset_key)
        fig.savefig(os.path.join(out_direc, out_fname), bbox_inches='tight')

    return fig, ax
    

In [ ]:
dset_keys = ['Masses', 'Metallicity', 'GrackleTemperature', 'Density', 'Pressure', 'CellSizes', 'JeansNumber']
for dset_key in dset_keys:
    addSFingGas = True
    fig, ax = plot_HistogramComparison(Sims, dset_key=dset_key, savefig=savefig, addSFingGas=addSFingGas)

In [ ]:
dicts = [dict(model='ST8', level=15),
         dict(model='ST8', level=14),
         dict(model='ST8', level=13),
         dict(model='TNG', level=15),
         dict(model='TNG', level=11)]

redshifts = [10.0, 7.0, 5.0, 4.0]

if savefig:

    for redshift_i, redshift in enumerate(redshifts):
        _Sims = []
        for _dict_i, _dict in enumerate(dicts):
            _Sims.append(Sim(**_dict, key='Redshift', val=redshift))
        fig, ax = plot_HistogramComparison(_Sims, dset_key='Metallicity', savefig=savefig, addSFingGas=addSFingGas)
    

In [ ]:
_Sim = Sims[0]
SubhaloGas = _Sim.SubhaloGas
SFR = SubhaloGas['StarFormationRate']
NJeans = SubhaloGas['JeansNumber']

mask = SFR > 0

fig, ax = plt.subplots()

h = ax.hist(NJeans[mask].value, bins=50, weights=SFR[mask]/SFR[mask].sum(), cumulative=True) 

ax.set_xlabel(r'Jeans Number [$N_J$]')
ax.set_ylabel(r'Cumulative Star Formation Rate $[\text{SFR}(<N_J) / \text{sum}(\text{SFR})]$')

fname = 'CumSFRNJeansDistribution_%s_z%.1f.pdf'%(_Sim.sim, _Sim.Redshift)
if savefig:
    fig.savefig(os.path.join(out_direc, fname), bbox_inches='tight')

In [ ]:
dset = Sims[0].SubhaloGas['GrackleCoolTime']
dset_key = 'GrackleCoolTime'
weights_key = 'Masses'

def plot_CoolTimeHistogram(Sims, dset_key='GrackleCoolTime', weights_key='Masses', addSFingGas=False, savefig=False):
    """Plot the 1D histogram of the cooling time, split into positive and negative cooling times """

    if isinstance(Sims, Sim):
        Sims = [Sims]

    bin_min = -0.5
    bin_max = 15.0
    nbins = 50
    bins = np.linspace(bin_min, bin_max, nbins)
    bin_width = bins[1] - bins[0]

    fig, ax = plt.subplots()

    counter = 0
    handles_hists = []
    handles_Sims = []
    for _Sim_i, _Sim in enumerate(Sims):
        _SubhaloGas = _Sim.SubhaloGas   
        if dset_key not in _SubhaloGas:
            if dset_key in dset_swap:
                dset_key = dset_swap[dset_key]
        _dset = _SubhaloGas[dset_key]

        # make extra histograms for only SFing Gas?
        masks = [np.ones_like(_dset.value, dtype=bool)]
        if addSFingGas:
            masks.append(_SubhaloGas['StarFormationRate'].value > 0)
            if _Sim_i == 0:
                handles_hists.append(ax.hist(np.zeros(0), bins=bins, histtype='stepfilled', color='tab:gray', label='SFing Gas', edgecolor='k')[2][0])

        for mask_i, mask in enumerate(masks):
            cooling_mask = _dset[mask].value < 0
            _weights = _SubhaloGas[weights_key][mask]
            dset_neg = np.log10(-1.0 * _dset[mask][cooling_mask].value)
            dset_pos = np.log10(_dset[mask][~cooling_mask].value)

            _kwargs = _Sim.kwargs.copy()
            kwargs = dict(histtype='step', linewidth=_kwargs['lw'], linestyle=_kwargs['ls'])

            if _Sim_i == 0:
                label_pos = 'Heating Up'
                label_neg = 'Cooling Down'
            else:
                label_pos = label_neg = None

            if mask_i == 1:
                kwargs.update(dict(alpha=0.25, histtype='stepfilled', hatch=_kwargs['hatch'], edgecolor='k'))
                label_pos = label_neg = None
            else:
                handles_Sims.append(ax.hist([], **kwargs, color='tab:gray', label=_Sim.plotTitle)[2][0])

            # offset bins slightly for visual clarity
            handles_hists.append(ax.hist(dset_pos, bins=bins+(0.1*bin_width)*counter, weights=_weights[~cooling_mask].value, color='tab:red', label=label_pos, **kwargs)[2][0])
            counter += 1
            handles_hists.append(ax.hist(dset_neg, bins=bins+(0.1*bin_width)*counter, weights=_weights[cooling_mask].value, color='tab:blue', label=label_neg, **kwargs)[2][0])
            counter += 1

    # only show legend entries for hists with a label 
    hanldes_hists_plot = []
    for handle in handles_hists:
        if str(handle.get_label()) != 'None':
            hanldes_hists_plot.append(handle)
    legend = ax.legend(handles=hanldes_hists_plot, title='All Subhalo Gas', loc='upper right')
    ax.add_artist(legend)

    ax.set_yscale('log')
    ax.set_ylabel(r'Gas Mass per %s bin [%s]'%(dset_key, _weights.unit.to_string('latex_inline')))
    ax.set_xlabel(r'%s [$\log_{10}$ %s]'%(dset_key, _dset.unit.to_string('latex_inline')))
    ax.set_ylim(5.0e1, 5.0e7)

    text = r'h%d $z=%.1f$'%(_Sim.targetHaloID, _Sim.Redshift)
    legend_Sims = ax.legend(handles=handles_Sims, title=text, loc='upper left')

    if savefig:
        if len(Sims) == 1:
            out_fname = 'CoolingTimeHistogram_%s_z%.1f.pdf'%(_Sim.sim, _Sim.Redshift)
        else:
            out_fname = 'CoolingTimeHistogramComparison_%s_h%d_z%s_z%.1f.pdf'%(_Sim.baseSim, _Sim.targetHaloID, _Sim.targetRedshift, _Sim.Redshift)
        fig.savefig(os.path.join(out_direc, out_fname), bbox_inches='tight')

    return fig, ax


In [ ]:
_Sims = []
for _Sim in Sims:
    if _Sim.model == 'ST8':
        _Sims.append(_Sim)
        fig, ax = plot_CoolTimeHistogram(_Sim, addSFingGas=True, savefig=savefig)

fig, ax = plot_CoolTimeHistogram(_Sims, addSFingGas=False, savefig=savefig)



### remake the figures here across resolution 

In [ ]:
_Sims = [L35n2160TNG_h31619_z3_L15_ST8, L35n2160TNG_h31619_z3_L14_ST8, L35n2160TNG_h31619_z3_L13_ST8,
         #L35n2160TNG_h31619_z3_L15_ST8m, L35n2160TNG_h31619_z3_L14_ST8m, L35n2160TNG_h31619_z3_L13_ST8m,
         L35n2160TNG_h31619_z3_L15_ST8s, L35n2160TNG_h31619_z3_L14_ST8s, L35n2160TNG_h31619_z3_L13_ST8s]

dset_key = 'AmbientDensity'
fkeys = ['sf_details', 'sn_details']

fig, axs = plt.subplots(2, 1, figsize=(9, 6), sharex=True)

bin_min = -6.0
bin_max = 12.0
nbins = 100
bins = np.linspace(bin_min, bin_max, nbins)
bin_width = bins[1] - bins[0]

mask_key = 'Time'
mask_val = 1. / (1. + 6.)

for _Sim in _Sims:

    for fkey_i, fkey in enumerate(fkeys):
        dic = _Sim.MCSTFiles[fkey]
        mask_dset = dic[mask_key]
        mask = mask_dset < mask_val
        dset = dic[dset_key][mask].copy()
        if 'Density' in dset_key:
            dset = dset.to(const.m_p / u.cm**3)
        vals = np.log10(dset.value)

        _kwargs = _Sim.kwargs.copy()
        kwargs = dict(histtype='step', color=_kwargs['c'], linewidth=_kwargs['lw'], linestyle=_kwargs['ls'], density=True)
        axs[fkey_i].hist(vals, bins=bins, **kwargs)

        if fkey_i == 0:
            axs[fkey_i].hist([], bins=bins, **kwargs, label=_Sim.plotTitle)

    
axs[1].set_xlabel(r'Ambient Gas Density $[\log_{10} (n = \rho / {\rm m_H}) \, {\rm cm^{-3}}]$')
axs[0].set_ylabel(r'PDF of Stars Formed per Density Bin')
axs[1].set_ylabel(r'PDF of Supernovae per Density Bin')
for ax in axs:
    ax.set_yscale('log')

title = r'h31619 $z \geq 6$ All SF and SN'
axs[0].legend(loc='upper left', title=title, ncol=2)

fname = 'AmbientDensitySFSNPDFs_Comparison_h31619_z8.pdf'
if savefig:
    fig.savefig(os.path.join(out_direc, fname), bbox_inches='tight')

In [ ]:
Densities = np.linspace(-8.0, 12.5, 100) * u.dex(u.cm**-3)
mu = 1.4
crit_size = 5. * (Densities.physical.value / mu)**(-0.46) * _Sim.Parameters['SNEnergyBoostFactor']**(0.29) * _Sim.Parameters['SNResolveFactor'] * u.pc

fig, ax = plt.subplots()
ax.set_yscale('log')
ax.set_xlabel(r'Gas Density $[\log_{10} (n = \rho / {\rm m_H}) \, {\rm cm^{-3}}]$')
ax.set_ylabel(r'Radius [pc]')

ax.plot(Densities, crit_size, 'k-', label='Critical Radius')

_Sims = [L35n2160TNG_h31619_z3_L15_ST8s,
         L35n2160TNG_h31619_z3_L14_ST8s]

for _Sim in _Sims:
    max_cell_size = (_Sim.Parameters['SNMaxCellRadius'] / _Sim.Header['HubbleParam'] * code_length).to('pc')
    bool_val = 'SN_MCS_RES_HOST' in _Sim.Config
    # if yes, then there may be a thermal dump in the host cell following Kim & Ostriker 2015
    TracerMass = (_Sim.Header['MassTable'][3] / _Sim.Header['HubbleParam'] * code_mass).to(standard_mass)
    TracerMCPerCell = _Sim.Parameters['TracerMCPerCell']
    TargetGasMass = TracerMass * TracerMCPerCell

    cell_size = ((TargetGasMass.to(const.m_p).value / mu / (4./3. * np.pi * Densities.physical))**(1/3)).to('pc')
    Density_max_cell_size = Densities[np.argmin(np.abs(max_cell_size - cell_size))]

    ax.plot(Densities, cell_size, color=_Sim.kwargs['color'], ls=_Sim.kwargs['ls'], marker='None', label=r'Cell Size at L%d'%_Sim.level)
    ax.axvline(Density_max_cell_size.value, color='tab:gray', ls=_Sim.kwargs['ls'], marker='None', label=r'SNMaxCellRadius at L%d'%_Sim.level)

ax.legend()

### make stellar light images of the group as it forms to identify intergalactic star-formation

In [ ]:
L35n2160TNG_h4182_z3_L14_ST8 = Sim(model='ST8', level=14, targetHaloID=4182, targetRedshift=3, key='Redshift', val=5.0, 
                                   addMCSTFiles=False, addSubhaloGas=False)
_Sim = L35n2160TNG_h4182_z3_L14_ST8
haloID = 0
snapNum = _Sim.snapNum

maxAgeYoungStars = 5 * u.Myr
SubhaloGasHalfmassRadFactor = 0.5

Header = _Sim.Header
HubbleParam = Header['HubbleParam']
cosmo = FlatLambdaCDM(H0=HubbleParam * 100.0, Om0=Header['Omega0'], Ob0=Header['OmegaBaryon'], Tcmb0=2.73)

_imgRmax = 125
nbins = 1024

string = 'GasSurfaceDensityMap'
_outdirec = os.path.join(out_direc, string, _Sim.sim)
if not os.path.isdir(_outdirec):
    os.makedirs(_outdirec)

vmin = 1.0e5 # Msun / kpc^2
vmax = 1.0e9 # Msun / kpc^2

cmap = mpl.cm.plasma.copy()
cmap.set_under(cmap(0.0), 1.0)
cmap.set_bad(cmap(0.0), 1.0)

minStellarMass = 0.01
minDMFraction = 0.1

cbar_kwargs = dict(pad=0.01, location='bottom', orientation='horizontal', fraction=0.15, aspect=20.,
                   label=r'Gas Surface Density [${\rm M_\odot} / {\rm pkpc}^2$]')
figSize = 4 # inches

for snapNum in _Sim.snapTimes['SnapNum'][69:70]:

    #createOffsets.createOffsets(_Sim.basePath, snapNum)

    Group = il.groupcat.loadSingle(_Sim.basePath, snapNum, haloID=haloID)
    io.convertGroupUnits(_Sim.basePath, snapNum, Group)
    GroupPos = Group['GroupPos']
    Subhalo = il.groupcat.loadSingle(_Sim.basePath, snapNum, subhaloID=int(Group['GroupFirstSub']))
    io.convertGroupUnits(_Sim.basePath, snapNum, Subhalo)
    SubhaloRmax = SubhaloGasHalfmassRadFactor * Subhalo['SubhaloHalfmassRadType'][0]

    Header = io.loadHeader(_Sim.basePath, snapNum)
    earliestCosmicTime = cosmo.age(Header['Redshift']).to('Myr') - maxAgeYoungStars
    earliestScaleFactor = 1.0 / (1.0 + z_at_value(cosmo.age, earliestCosmicTime)).value

    imgRmax = _imgRmax * Header['Time']
    offset = imgRmax / 10. 
    length = imgRmax / 5.
    bin_width = 2.0 * imgRmax / nbins

    # find recently formed stars
    stars = il.snapshot.loadSubhalo(_Sim.basePath, snapNum, int(Group['GroupFirstSub']), 'star', fields=['Coordinates', 'StellarFormationTime', 'Masses'])
    if stars['count'] == 0:
        continue
    #gas = il.snapshot.loadHalo(_Sim.basePath, snapNum, haloID, 'gas', fields=['Masses', 'Coordinates', 'Density'])
    gas = il.snapshot.loadSubset(_Sim.basePath, snapNum, 'gas', fields=['Masses', 'Coordinates', 'Density', 'HighResGasMass'])
    if gas['count'] == 0:
        continue
    
    io.convertSnapshotUnits(_Sim.basePath, snapNum, stars)
    io.convertSnapshotUnits(_Sim.basePath, snapNum, gas)
    
    # only consider high res gas cells
    mask = gas['HighResGasMass']/gas['Masses'] > 0.5
    for key in gas:
        if key == 'count':
            gas[key] = mask[mask].size 
        else:
            gas[key] = gas[key][mask]

    fig, ax = plt.subplots(figsize=(figSize, figSize*(1 + cbar_kwargs['pad'] + cbar_kwargs['fraction'])))
    ax.set_xticks([])
    ax.set_yticks([])

    c = 'white'
    ax.plot([(-imgRmax + offset), (-imgRmax + offset + length)], [(imgRmax - offset), (imgRmax - offset)],
            marker='None', ls='-', lw=4, c=c)
    ax.text(((-imgRmax + offset) + (-imgRmax + offset + length))/2, imgRmax - 2*offset, r'%.1f pkpc'%(length),
            ha='center', va='top', c=c, fontsize=12)
    ax.text(0.975, 0.975, r'$z = %.2f$'%Header['Redshift'], va='top', ha='right', transform=ax.transAxes, fontsize=12, c=c)

    text = (r'${\rm h%d\_L%d\_%s}$ snap %03d'%(_Sim.targetHaloID, _Sim.level, _Sim.model, snapNum) + '\n' + 
            r'$R_{\rm half,\star} = %.1f,\ R_{\rm half,gas} = %.1f$ pkpc'%(Subhalo['SubhaloHalfmassRadType'][4].value, Subhalo['SubhaloHalfmassRadType'][0].value) + '\n' + 
            r'$M_{\rm \star} = %.1f,\ M_{\rm 200c} = %.1f\, \log_{10} {\rm M_\odot}$'%(np.log10(Subhalo['SubhaloMassInRadType'][4].value), np.log10(Group['Group_M_Crit200'].value)))
    ax.text(0.975, 0.025, text, ha='right', va='bottom', ma='right', transform=ax.transAxes, color=c)

    BoxSize = _Sim.Header['BoxSize'] * Header['Time'] / HubbleParam * u.kpc
    starsCoordinates = ru.shift(stars['Coordinates'], GroupPos, BoxSize)
    starsDistances = ru.mag(starsCoordinates, np.zeros(3) * u.kpc, BoxSize) 

    starsMask = ((stars['StellarFormationTime'] >= earliestScaleFactor) &
                 (starsDistances > SubhaloRmax))
    
    ax.plot(starsCoordinates[:,0][starsMask].value, starsCoordinates[:,1][starsMask].value, marker='o', c=c, fillstyle='full', ls='None', ms=1.0)

    # mark the locations of satellites
    subhalos = il.groupcat.loadSubhalos(_Sim.basePath, snapNum, fields=['SubhaloPos', 'SubhaloMass', 'SubhaloMassInRadType', 'SubhaloGrNr', 'SubhaloMassType'])
    io.convertGroupUnits(_Sim.basePath, snapNum, subhalos)
    maskSubhaloGrNr = subhalos['SubhaloGrNr'] == 0
    maskMergerRatio = (subhalos['SubhaloMassInRadType'][:,4] / Subhalo['SubhaloMassInRadType'][4]) > minStellarMass
    maskDMFraction = (subhalos['SubhaloMassType'][:,il.util.partTypeNum('dm')] / subhalos['SubhaloMass']) > minDMFraction
    mask = maskSubhaloGrNr & maskMergerRatio & maskDMFraction
    subhaloIDs = np.where(mask)[0]
    subhalosPos = ru.shift(subhalos['SubhaloPos'][mask], GroupPos, BoxSize)
    ax.plot(subhalosPos[:,0], subhalosPos[:,1], color=c, marker='x', ms=10)

    gasCoordinates = ru.shift(gas['Coordinates'], GroupPos, BoxSize).value
    gasMasses = gas['Masses'].value
    io.computeCellSizes(gas, _Sim.basePath, snapNum)
    gasSizes = gas['CellSizes'].value

    gasMask = (np.abs(gasCoordinates[:,2]) < imgRmax)

    h = sphMap(gasCoordinates, 1.5 * gasSizes, gasMasses, None, [0,1], [2*imgRmax]*3, [BoxSize.value]*3, [0]*3, [nbins]*2, 3, colDens=True)

    img = ax.imshow(h, extent=[-imgRmax, imgRmax, -imgRmax, imgRmax], origin='lower', norm=mpl.colors.LogNorm(vmin=vmin, vmax=vmax), cmap=cmap)

    cbar = fig.colorbar(img, **cbar_kwargs)

    fname = 'snapNum%03d.png'%(snapNum)
    if savefig:
        fig.savefig(os.path.join(_outdirec, fname), bbox_inches='tight', dpi=300)
        plt.close(fig)


In [ ]:
_Sim = L35n2160TNG_h4182_z3_L14_ST8
haloID = 0
snapNum = _Sim.snapNum

minStellarMass = 0.01 # ratio wrt central subahlo
minDMFraction = 0.1 # wrt Subhalo Mass
minSubhaloMass = 1.0e8 * u.M_sun 
maxDistance = 300 * u.kpc # ckpc

r = {}

for snapNum in _Sim.snapTimes['SnapNum']:

    r[snapNum] = {}

    Header = io.loadHeader(_Sim.basePath, snapNum)

    BoxSize = Header['BoxSize'] * Header['Time'] / Header['HubbleParam'] * u.kpc
    
    Group = il.groupcat.loadSingle(_Sim.basePath, snapNum, haloID=haloID)
    io.convertGroupUnits(_Sim.basePath, snapNum, Group)
    GroupPos = Group['GroupPos']

    subhalos = il.groupcat.loadSubhalos(_Sim.basePath, snapNum, fields=['SubhaloPos', 'SubhaloMass', 'SubhaloMassInRadType', 'SubhaloGrNr', 'SubhaloMassType'])
    io.convertGroupUnits(_Sim.basePath, snapNum, subhalos)
    maskSubhaloGrNr = subhalos['SubhaloGrNr'] == 0
    maskDistance = ru.mag(subhalos['SubhaloPos'], GroupPos, BoxSize) < (maxDistance * Header['Time'])
    maskMergerRatio = (subhalos['SubhaloMassInRadType'][:,4] / Subhalo['SubhaloMassInRadType'][4]) > minStellarMass
    maskSubhaloMass = (subhalos['SubhaloMass']) > minSubhaloMass
    maskDMFraction = (subhalos['SubhaloMassType'][:,il.util.partTypeNum('dm')] / subhalos['SubhaloMass']) > minDMFraction
    mask = maskSubhaloGrNr & maskDistance & maskMergerRatio & maskDMFraction & maskSubhaloMass
    subhaloIDs = np.where(mask)[0]
    subhalosPos = ru.shift(subhalos['SubhaloPos'][mask], GroupPos, BoxSize)

    r[snapNum]['SatelliteMagsPhys'] = ru.mag(subhalosPos[1:], np.zeros(3), BoxSize)
    r[snapNum]['SatelliteMagsCom'] = r[snapNum]['SatelliteMagsPhys'] / Header['Time']
    r[snapNum]['SubfindIDs'] = subhaloIDs[1:]


In [ ]:
Header = _Sim.Header
HubbleParam = Header['HubbleParam']
cosmo = FlatLambdaCDM(H0=HubbleParam * 100.0 * u.km / u.s / u.Mpc, Om0=Header['Omega0'], Ob0=Header['OmegaBaryon'], Tcmb0=2.73 * u.K)

startRedshift = 20
stopRedshift = 3
nRedshift = int(1e4)
redshifts = np.linspace(startRedshift, stopRedshift, nRedshift)
cosmic_times = cosmo.age(redshifts)
interpCosmicTimes = interp1d(redshifts, cosmic_times, kind='cubic')
interpRedshifts = interp1d(cosmic_times, redshifts, kind='cubic')

fig, ax = plt.subplots()

cmap = mpl.cm.tab10.copy()
vmin = 0.5
vmax = 10.5

for snap in r:
    mags = r[snap]['SatelliteMagsCom']
    subfindIDs = r[snap]['SubfindIDs']
    sc = ax.scatter([_Sim.snapTimes['CosmicTime'][snap]]*len(mags), mags, marker='.', c=subfindIDs, cmap=cmap, vmin=vmin, vmax=vmax)

ax.set_yscale('log')
ax.set_xlabel(r'Cosmic Time [Gyr]')
ax.set_ylabel(r'Host-Centric Distance [ckpc]')

cbar_kwargs = dict(pad=0.01, location='right', orientation='vertical', fraction=0.15, aspect=20.,
                   label=r'Subfind ID', ticks=np.arange(int(round(vmin+0.5)), int(round(vmax+0.5))))

cbar = fig.colorbar(sc, **cbar_kwargs)
cbar.ax.minorticks_off()

text = ('h%d_L%d_%s'%(_Sim.targetHaloID, _Sim.level, _Sim.model) + '\n' + 
        'FoF Satellites with' + '\n' + 
        r'$M_{\star}^{\rm sat} / M_{\star}^{\rm cen} > %.2f$'%minStellarMass + '\n' + 
        r'$M_{\rm tot}^{\rm sat} > 10^8$ %s'%(minSubhaloMass.unit.to_string('latex_inline')) + '\n'
        r'$M_{\rm dm}^{\rm sat} / M_{\rm tot}^{\rm sat} > %.1f$'%minDMFraction + '\n' + 
        r'$d_{\rm sat}^{\rm cen} < 300$ ckpc')

ax.text(0.025, 0.025, text, ma='left', ha='left', va='bottom', transform=ax.transAxes)

# Create a second x-axis for redshift
ax2 = ax.twiny()

# Set the limits of the second x-axis to match the primary x-axis
ax2.set_xlim(ax.get_xlim())

# Generate the redshift values corresponding to the cosmic time values
redshift_values = [10, 8, 7, 6, 5, 4, 3.5, 3]

# Set the ticks and labels for the second x-axis
ax2.set_xticks(interpCosmicTimes(redshift_values))
ax2.set_xticklabels([f'{z:.1f}' for z in redshift_values])
ax2.tick_params(axis='x', which='minor', top=False)
ax2.set_xlabel('Redshift')

if savefig:
    fname = '%s_SatelliteDistancesEvolution.pdf'%_Sim.sim
    fig.savefig(os.path.join(out_direc, fname), bbox_inches='tight')


In [ ]:
snapNum = 174

_imgRmax = 20
nbins = 256

cmap = mpl.cm.bone.copy()
cmap.set_under(cmap(0.0), 1.0)
cmap.set_bad(cmap(0.0), 1.0)
vmin = 1.0e2 # Msun / kpc^2
vmax = 1.0e8 # Msun / kpc^2

cbar_kwargs = dict(pad=0.01, location='bottom', orientation='horizontal', fraction=0.15, aspect=20.,
                   label=r'Stellar Surface Density [${\rm M_\odot} / {\rm pkpc}^2$]')

Group = il.groupcat.loadSingle(_Sim.basePath, snapNum, haloID=haloID)
io.convertGroupUnits(_Sim.basePath, snapNum, Group)
GroupPos = Group['GroupPos']
Subhalo = il.groupcat.loadSingle(_Sim.basePath, snapNum, subhaloID=int(Group['GroupFirstSub']))
io.convertGroupUnits(_Sim.basePath, snapNum, Subhalo)
SubhaloRmax = SubhaloGasHalfmassRadFactor * Subhalo['SubhaloHalfmassRadType'][0]

Header = io.loadHeader(_Sim.basePath, snapNum)

imgRmax = _imgRmax * Header['Time']
bins = np.linspace(-imgRmax, imgRmax, num=nbins)
offset = imgRmax / 10. 
length = imgRmax / 5.
bin_width = bins[1] - bins[0]

stars = il.snapshot.loadSubhalo(_Sim.basePath, snapNum, int(Group['GroupFirstSub']), 'star', fields=['Coordinates', 'StellarFormationTime', 'Masses'])
io.convertSnapshotUnits(_Sim.basePath, snapNum, stars)

fig, ax = plt.subplots(figsize=(figSize, figSize*(1 + cbar_kwargs['pad'] + cbar_kwargs['fraction'])))
ax.set_xticks([])
ax.set_yticks([])

c = 'white'

ax.plot([(-imgRmax + offset), (-imgRmax + offset + length)], [(imgRmax - offset), (imgRmax - offset)],
        marker='None', ls='-', lw=4, c=c)
ax.text(((-imgRmax + offset) + (-imgRmax + offset + length))/2, imgRmax - 2*offset, r'%.1f pkpc'%(length),
        ha='center', va='top', c=c, fontsize=12)
ax.text(0.975, 0.975, r'$z = %.2f$'%Header['Redshift'], va='top', ha='right', transform=ax.transAxes, fontsize=12, c=c)

starsCoordinates = ru.shift(stars['Coordinates'], GroupPos, BoxSize).value
starsMasses = stars['Masses'].value

starsMask = (np.abs(starsCoordinates[:,2]) < imgRmax)

h = np.histogram2d(starsCoordinates[:,0], starsCoordinates[:,1], bins, weights=starsMasses)[0]

img = ax.imshow(h, extent=[-imgRmax, imgRmax, -imgRmax, imgRmax], origin='lower', norm=mpl.colors.LogNorm(vmin=vmin, vmax=vmax), cmap=cmap)

cbar = fig.colorbar(img, **cbar_kwargs)


In [ ]:
Header = _Sim.Header
HubbleParam = Header['HubbleParam']
cosmo = FlatLambdaCDM(H0=HubbleParam * 100.0 * u.km / u.s / u.Mpc, Om0=Header['Omega0'], Ob0=Header['OmegaBaryon'], Tcmb0=2.73 * u.K)

startRedshift = 50
stopRedshift = 2.5
nRedshift = int(1e4)
redshifts = np.linspace(startRedshift, stopRedshift, nRedshift)
cosmic_times = cosmo.age(redshifts)
interpCosmicTimes = interp1d(redshifts, cosmic_times, kind='cubic')
interpRedshifts = interp1d(cosmic_times, redshifts, kind='cubic')



In [ ]:
stars = il.snapshot.loadSubhalo(_Sim.basePath, snapNum, int(Group['GroupFirstSub']), 'star', fields=['Coordinates', 'StellarFormationTime', 'Masses'])
io.convertSnapshotUnits(_Sim.basePath, snapNum, stars)
starsBirthTimes = interpCosmicTimes(1. / stars['StellarFormationTime'] - 1) * u.Gyr

In [ ]:
fig, ax = plt.subplots()

bin_start = 0 * u.Gyr
bin_stop = 2.25 * u.Gyr
nbins = 100
bins = np.linspace(bin_start, bin_stop, nbins)
bin_width = (bins[1] - bins[0]).to('yr')
bin_cents = bins[:-1] + bin_width / 2.

r = np.zeros(nbins-1) - 1
h = np.histogram(starsBirthTimes, bins=bins, weights=stars['Masses'])[0] / bin_width
h_smoothed = gaussian_filter(h.value, 1.0) * h.unit

x_plot = bin_cents[h > 0]
y_plot = h_smoothed[h > 0]

ax.plot(x_plot, y_plot, 'b--', marker='None')

ax.set_yscale('log')
ax.set_ylim(1.0e-2, 2.0e1)
ax.set_xlim()

ax.set_xlabel(r'Cosmic Time [Gyr]')
ax.set_ylabel(r'Star Formation Rate $[\rm{M_\odot / yr}]$')


twiny = True
if twiny:
    # Create a second x-axis for redshift
    ax2 = ax.twiny()

    # Set the limits of the second x-axis to match the primary x-axis
    ax2.set_xlim(ax.get_xlim())

    # Generate the redshift values corresponding to the cosmic time values
    redshift_values = [15, 10, 8, 7, 6, 5, 4, 3.5, 3]

    # Set the ticks and labels for the second x-axis
    ax2.set_xticks(interpCosmicTimes(redshift_values))
    ax2.set_xticklabels([f'{z:.1f}' for z in redshift_values])

    ax2.tick_params(axis='x', which='minor', top=False)
else:
    ax2 = ax.secondary_xaxis('top', functions=(interpRedshifts, interpCosmicTimes))

ax2.set_xlabel('Redshift')

